# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [24]:
!pip install pandas numpy scikit-learn gensim transformers torch

import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
df = pd.read_csv("data/reviews_data.csv")
df = df.sample(2000, random_state=42)

df = df[['review_body', 'stars']].dropna()

df['review_body'] = df['review_body'].str.lower()

df.head()

,review_body,stars
13806,i love it😚😚😚😚😚,5
5852,i had a dispute on my account and within 5 day...,5
11170,so my account with chime was closed january of...,1
11097,there is no concern whatsoever for fraudulent ...,1
13581,👍 have been good,5


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    df['review_body'], df['stars'], test_size=0.2, random_state=42)

In [28]:
def tokenize(text):
    return re.findall(r'\b\w+\b', text)

X_train_tokens = X_train.apply(tokenize)
X_test_tokens = X_test.apply(tokenize)

In [29]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [30]:
def get_avg_embedding(tokens, model, size=100):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(size)
    return np.mean(vectors, axis=0)

X_train_vec = np.array([get_avg_embedding(tokens, w2v_model) for tokens in X_train_tokens])
X_test_vec = np.array([get_avg_embedding(tokens, w2v_model) for tokens in X_test_tokens])

In [31]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec, y_train)

y_pred_w2v = clf.predict(X_test_vec)

print("Word2Vec Accuracy:", accuracy_score(y_test, y_pred_w2v))
print(classification_report(y_test, y_pred_w2v))

Word2Vec Accuracy: 0.5981873111782477
              precision    recall  f1-score   support

           1       0.00      0.00      0.00        86
           2       0.00      0.00      0.00        11
           3       0.00      0.00      0.00        14
           4       0.00      0.00      0.00        22
           5       0.60      1.00      0.75       198

    accuracy                           0.60       331
   macro avg       0.12      0.20      0.15       331
weighted avg       0.36      0.60      0.45       331



C:\Users\sbasd\PycharmProjects\JupyterProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sbasd\PycharmProjects\JupyterProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\sbasd\PycharmProjects\JupyterProject\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [32]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import torch

In [33]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [34]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=64,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx] - 1)  # shift to 0–4
        }

In [35]:
train_dataset = ReviewDataset(X_train, y_train)
test_dataset = ReviewDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [36]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=5
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4567.01it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [40]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(5):
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        if i % 50 == 0:
            print(f"Batch {i}, Loss: {loss.item()}")

    print(f"Epoch {epoch+1} complete")

Batch 0, Loss: 0.2319498509168625
Batch 50, Loss: 0.5165414810180664
Batch 100, Loss: 0.10292205214500427
Batch 150, Loss: 0.4430672824382782
Epoch 1 complete
Batch 0, Loss: 0.6068223118782043
Batch 50, Loss: 0.12829476594924927
Batch 100, Loss: 0.2214422971010208
Batch 150, Loss: 0.32178300619125366
Epoch 2 complete
Batch 0, Loss: 0.1834770292043686
Batch 50, Loss: 0.4741225838661194
Batch 100, Loss: 0.2067713737487793
Batch 150, Loss: 0.015289348550140858
Epoch 3 complete
Batch 0, Loss: 0.018416956067085266
Batch 50, Loss: 0.03832228481769562
Batch 100, Loss: 0.19402232766151428
Batch 150, Loss: 0.1758110523223877
Epoch 4 complete
Batch 0, Loss: 0.0071836900897324085
Batch 50, Loss: 0.09059780091047287
Batch 100, Loss: 0.3246152698993683
Batch 150, Loss: 0.14496763050556183
Epoch 5 complete


In [41]:
model.eval()

preds, true = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        predictions = torch.argmax(logits, dim=1).cpu().numpy()
        labels = batch['labels'].cpu().numpy()

        preds.extend(predictions)
        true.extend(labels)

# convert back to 1–5
preds = [p + 1 for p in preds]
true = [t + 1 for t in true]

print("BERT Accuracy:", accuracy_score(true, preds))
print(classification_report(true, preds))

BERT Accuracy: 0.7854984894259819
              precision    recall  f1-score   support

           1       0.88      0.79      0.83        86
           2       0.25      0.45      0.32        11
           3       0.17      0.14      0.15        14
           4       0.22      0.27      0.24        22
           5       0.92      0.90      0.91       198

    accuracy                           0.79       331
   macro avg       0.49      0.51      0.49       331
weighted avg       0.81      0.79      0.80       331

